# Vintage Diagnostics (pre-Phase 5 correction)

Measures three parameters currently assumed rather than measured:

1. **Is the 2016 OOT set trustworthy?** — observed vs lower-bound default rate per vintage.
2. **Where should the train window start?** — first year each bureau feature is actually collected.
3. **What outcome horizon H?** — month-on-book by which ~90% of a vintage's defaults have resolved.

Runs on the RAW ingested frame. `build_target()` must NOT run first: it drops exactly the
censored rows this notebook needs to count.


In [1]:
from pathlib import Path

import polars as pl

from credit_risk.data.ingestion import load_raw_accepted_loans
from credit_risk.data.diagnostics import (
    censoring_by_vintage,
    default_hazard_by_mob,
    feature_availability,
    first_reliable_year,
    prepayment_risk_link,
)

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(40)

DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")
raw = load_raw_accepted_loans(DATA_PATH)
print(raw.shape)


(2260668, 151)


## 1. Censoring bias per vintage

`dr_observed` is what `build_target()` currently reports (defaults / matured).
`dr_lower_bnd` is defaults / issued — the true rate if no censored loan ever defaults.
The true vintage rate lies between them; `bias_gap` is the size of the distortion.

**Decision rule:** any vintage whose `bias_gap` exceeds ~2pp cannot be used as an
evaluation set under the current target definition.


In [2]:
censoring = censoring_by_vintage(raw)
censoring


issue_year,n_issued,n_matured,n_default,pct_censored,dr_observed,dr_lower_bnd,bias_gap
i32,u32,u32,u32,f64,f64,f64,f64
2007,603,603,158,0.0,0.262023,0.262023,0.0
2008,2393,2393,496,0.0,0.207271,0.207271,0.0
2009,5281,5281,723,0.0,0.136906,0.136906,0.0
2010,12537,12537,1757,0.0,0.140145,0.140145,0.0
2011,21721,21721,3297,0.0,0.151789,0.151789,0.0
2012,53367,53367,8644,0.0,0.161973,0.161973,0.0
2013,134814,134804,21024,0.000074,0.15596,0.155948,0.000012
2014,235629,223103,41162,0.05316,0.184498,0.17469,0.009808
2015,421095,375546,75804,0.108168,0.20185,0.180016,0.021834


In [3]:
# Export for review alongside this notebook.
censoring.write_csv("../docs/vintage_censoring.csv")


## 2. Feature availability per vintage

LendingClub expanded bureau collection in stages. A feature whose null rate collapses
from ~1.0 to ~0.0 at some year is vintage dependent: its WOE `missing` bin encodes
calendar time, not credit risk, and that bin is never hit at serving time.

Half of `SCORECARD_FEATURES` is suspected to be in this group.


In [4]:
SUSPECT_FEATURES = [
    # kept in SCORECARD_FEATURES, suspected post-2012 bureau expansion
    "acc_open_past_24mths", "mort_acc", "mths_since_recent_bc", "avg_cur_bal",
    "bc_open_to_buy", "num_rev_tl_bal_gt_0", "mo_sin_rcnt_tl", "mths_since_recent_inq",
    # control group: expected available from 2007
    "fico_range_low", "dti", "annual_inc", "revol_util", "open_acc",
    # opposite direction: suspected to start only in 2016 (absent in train)
    "disbursement_method", "initial_list_status",
]

availability = feature_availability(raw, SUSPECT_FEATURES)
availability


issue_year,n,acc_open_past_24mths,mort_acc,mths_since_recent_bc,avg_cur_bal,bc_open_to_buy,num_rev_tl_bal_gt_0,mo_sin_rcnt_tl,mths_since_recent_inq,fico_range_low,dti,annual_inc,revol_util,open_acc,disbursement_method,initial_list_status
i32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2007,603,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.006633,0.053068,0.048093,0.0,0.0
2008,2393,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.005015,0.0,0.0,0.0
2009,5281,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.00303,0.0,0.0,0.0
2010,12537,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.001675,0.0,0.0,0.0
2011,21721,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.000414,0.0,0.0,0.0
2012,53367,0.140443,0.140443,0.148931,0.519816,0.150261,0.519816,0.519816,0.248955,0.0,0.0,0.0,0.000881,0.0,0.0,0.0
2013,134814,0.0,0.0,0.006528,0.000045,0.007462,0.0,0.0,0.108164,0.0,0.0,0.0,0.000579,0.0,0.0,0.0
2014,235629,0.0,0.0,0.009532,0.000025,0.010376,0.0,0.0,0.09206,0.0,0.0,0.0,0.00053,0.0,0.0,0.0
2015,421095,0.0,0.0,0.009019,0.0,0.009411,0.0,0.0,0.105912,0.0,0.000005,0.0,0.000385,0.0,0.0,0.0


In [5]:
first_reliable_year(availability, max_null_rate=0.05)


feature,first_year
str,i32
"""annual_inc""",2007
"""disbursement_method""",2007
"""dti""",2007
"""fico_range_low""",2007
"""initial_list_status""",2007
"""open_acc""",2007
"""revol_util""",2008
"""acc_open_past_24mths""",2013
"""avg_cur_bal""",2013


In [6]:
availability.write_csv("../docs/vintage_feature_availability.csv")


## 3. Default timing — choosing the outcome horizon H

Run on fully matured vintages only (2012–2013). Charge-off date is not published by
LendingClub, so it is proxied as `last_pymnt_d + 5` months (LC charges off at ~121 days
delinquent).

**Two things to read here:**
- *Proxy validity:* `n_default` should peak around MOB 8–18 and decline. A flat or
  bimodal shape means the proxy is wrong and H cannot be set from it.
- *Horizon:* the MOB where `cum_share` reaches ~0.90, separately for 36 and 60 month terms.


In [7]:
hazard = default_hazard_by_mob(raw, vintages=[2012, 2013])

for term in sorted(hazard["term_months"].unique()):
    sub = hazard.filter(pl.col("term_months") == term)
    for q in (0.50, 0.75, 0.90, 0.95):
        mob = sub.filter(pl.col("cum_share") >= q)["mob"].min()
        print(f"term={term}m  {q:.0%} of defaults resolved by MOB {mob}")
    print()


term=36m  50% of defaults resolved by MOB 22
term=36m  75% of defaults resolved by MOB 29
term=36m  90% of defaults resolved by MOB 35
term=36m  95% of defaults resolved by MOB 37

term=60m  50% of defaults resolved by MOB 27
term=60m  75% of defaults resolved by MOB 39
term=60m  90% of defaults resolved by MOB 49
term=60m  95% of defaults resolved by MOB 55



In [8]:
hazard.filter(pl.col("term_months") == 36).head(40)


term_months,mob,n_default,cum_share
i32,i32,u32,f64
36,6,179,0.009862
36,7,297,0.026226
36,8,369,0.046556
36,9,447,0.071185
36,10,465,0.096804
36,11,544,0.126777
36,12,619,0.160882
36,13,633,0.195758
36,14,688,0.233664


## 4. Does censoring also inflate measured AUC?

Censoring inflates the default rate unconditionally. It inflates *discrimination* only
if who-prepays correlates with risk — in that case the matured subset is a risk-widened
sample and its AUC is optimistic too.

**Decision rule:** if early payers average more than ~10 FICO points above on-schedule
payers, the reported OOT AUC of 0.721 is optimistic, not just the OOT default rate.


In [9]:
prepay = prepayment_risk_link(raw, vintages=[2012, 2013])
prepay


term_months,early_payer,n,mean_fico,mean_dti
i32,bool,u32,f64,f64
36,false,63849,697.945074,16.921249
36,true,61762,698.229332,16.092545
60,false,9146,699.214411,18.771905
60,true,23746,698.683778,17.558057


In [10]:
fico_gap = (
    prepay.pivot(on="early_payer", index="term_months", values="mean_fico")
    .with_columns((pl.col("true") - pl.col("false")).alias("fico_gap_early_minus_scheduled"))
)
fico_gap


term_months,false,true,fico_gap_early_minus_scheduled
i32,f64,f64,f64
36,697.945074,698.229332,0.284258
60,699.214411,698.683778,-0.530632


## Summary — parameters this notebook decides

| Parameter | Source | Value |
|---|---|---|
| 2016 OOT usable as-is? | section 1, `bias_gap` | |
| `train_start` (new) | section 2, `first_reliable_year` | |
| Horizon H, term 36 | section 3, `cum_share` >= 0.90 | |
| Horizon H, term 60 | section 3, `cum_share` >= 0.90 | |
| OOT AUC also inflated? | section 4, `fico_gap` | |

Fill this in, then it feeds directly into: rewriting `data/target.py` to a fixed
outcome window, shifting `split.train_end`'s lower bound in `configs/base.yaml`,
and re-running every number in `docs/modeling_findings.md`.


In [11]:
from credit_risk.data.diagnostics import charge_off_proxy_check

proxy = charge_off_proxy_check(raw, vintages=[2012, 2013])
proxy.write_csv("../docs/charge_off_proxy_check.csv")
proxy

has_recovery,n,med_mob_last_pymnt,med_n_pymt_implied,med_gap,gap_p25,gap_p75
bool,u32,f64,f64,f64,f64,f64
false,1946,20.0,20.0,0.0,0.0,0.949993
true,27572,18.0,17.971186,0.012246,0.001482,1.0
